In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [3]:
# Uninstall unused TensorFlow to prevent protobuf clash
!pip uninstall -y tensorflow tensorflow-intel
!pip install -q --upgrade "protobuf>=4.25.0"
# Ensure transformers, sentencepiece, and evaluation libraries are installed
!pip install -q --upgrade transformers sentencepiece sacrebleu evaluate indic-nlp-library


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires huggingface-hub<1.0,>=0.19.3, but you have huggingface-hub 1.31.0 which is incompatible.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 7.36.1 which is incompatible.
unbabel-comet 2.2.7 requires transformers<5.0,>=4.17, but you have transformers 5.17.0 which is incompatible.


In [4]:
import os
# Disable TensorFlow detection so transformers stays pure PyTorch
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

!python bhashaverse_translate.py


Loading SPM model …
Loading fairseq dictionary …
Loading HF model …
config.json: 100% 925/925 [00:00<00:00, 2.65MB/s]
model.safetensors: 100% 4.41G/4.41G [00:57<00:00, 76.0MB/s]
Loading weights: 100% 771/771 [00:00<00:00, 4752.53it/s]
generation_config.json: 100% 160/160 [00:00<00:00, 547kB/s]
Ready on cpu  (1,101,201,408 parameters)
Hello, how are you?
  BhashaVerse (IIIT Hyderabad) Local Batch Inference & Evaluation
  Model Repository: ltrciiith/bhashaverse (mBART-based OneMT v3b architecture)
Device: cpu

Processing dataset: /content/drive/MyDrive/Annotated Data - Filtered Dataset.csv
Output directory  : /content/drive/MyDrive/Inference final/BhashaVerse/Annotated_Filtered
Loaded 100 valid sentence pairs (from 101 raw rows).
Sample English  : This book, Finding Darwin's God, by Kenneth Miller, is one of the most effe...
Sample Malayalam: ഈ കാണുന്ന, കെന്നത്ത് മില്ലര് എഴുതിയ ഫയ്ന്റ്റിംഗ് ഡാര് വിന് സ് ഗോഡ് എന്ന പുസ...
Remaining sentences to translate: 100
   Batch  1 Done [  6/100 (  6

In [5]:
# ============================================================
# Evaluation + Save Metrics Permanently
# ============================================================
import os
import json
import torch
import pandas as pd
import sacrebleu
import evaluate
import nltk
from indicnlp.tokenize import indic_tokenize

# Locate predictions CSV generated by bhashaverse_translate.py
PRED_CANDIDATES = [
    "/content/drive/MyDrive/Inference final/BhashaVerse/Annotated_Filtered/predictions.csv",
    os.path.join(os.getcwd(), "Inference final", "BhashaVerse", "Annotated_Filtered", "predictions.csv"),
    "/content/drive/MyDrive/Inference final/BhashaVerse/IN22-Gen/predictions.csv",
]

PRED_CSV = None
for p in PRED_CANDIDATES:
    if os.path.exists(p):
        PRED_CSV = p
        break

if PRED_CSV is None:
    raise FileNotFoundError(
        f"Predictions CSV not found. Please ensure 'bhashaverse_translate.py' ran successfully. Checked: {PRED_CANDIDATES}"
    )

print(f"Loading predictions from: {PRED_CSV}")
df = pd.read_csv(PRED_CSV)

# Detect columns flexibly
eng_col = next((c for c in df.columns if c.strip().lower() in ["english sentence", "english", "eng_latn", "src"]), None)
ref_col = next((c for c in df.columns if c.strip().lower() in ["malayalam sentence", "malayalam", "reference_malayalam", "tgt"]), None)
pred_col = next((c for c in df.columns if c.strip().lower() in ["bhashaverse_malayalam", "model_malayalam", "predicted_malayalam", "prediction"]), None)

if not all([eng_col, ref_col, pred_col]):
    raise ValueError(f"Could not resolve columns in {PRED_CSV}. Columns found: {list(df.columns)}")

df = df.dropna(subset=[eng_col, pred_col]).copy()
en_sents = df[eng_col].astype(str).str.strip().tolist()
reference = df[ref_col].astype(str).str.strip().tolist()
predictions = df[pred_col].astype(str).str.strip().tolist()

print(f"Evaluating {len(predictions)} sentence pairs...")

# ------------------------------------------------------------
# 1. SacreBLEU (Standard 13a Tokenizer)
# Note: sacrebleu expects references as a list of reference streams: [reference]
# ------------------------------------------------------------
sacrebleu_13a = sacrebleu.corpus_bleu(
    predictions,
    [reference],
    tokenize="13a"
).score

# ------------------------------------------------------------
# 2. Indic-Tokenized SacreBLEU (AI4Bharat / IndicTrans2 Standard)
# Pre-tokenizing Malayalam with IndicNLP handles complex script & morphemes
# ------------------------------------------------------------
preds_indic_tok = [" ".join(indic_tokenize.trivial_tokenize(p, lang="ml")) for p in predictions]
refs_indic_tok = [" ".join(indic_tokenize.trivial_tokenize(r, lang="ml")) for r in reference]

sacrebleu_indic = sacrebleu.corpus_bleu(
    preds_indic_tok,
    [refs_indic_tok],
    tokenize="none"
).score

# ------------------------------------------------------------
# 3. SacreBLEU (FLORES-200 Tokenizer if supported)
# ------------------------------------------------------------
try:
    sacrebleu_flores = sacrebleu.corpus_bleu(
        predictions,
        [reference],
        tokenize="flores200"
    ).score
except Exception:
    try:
        sacrebleu_flores = sacrebleu.corpus_bleu(
            predictions,
            [reference],
            tokenize="flores101"
        ).score
    except Exception:
        sacrebleu_flores = None

# ------------------------------------------------------------
# 4. HuggingFace Evaluate BLEU
# evaluate.load('bleu') expects references as [[r1], [r2], ...]
# ------------------------------------------------------------
bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=[[x] for x in reference]
)["bleu"]

# ------------------------------------------------------------
# 5. chrF and chrF++ (Character n-gram F-score; Primary for Indic MT)
# ------------------------------------------------------------
chrf_score = sacrebleu.corpus_chrf(
    predictions,
    [reference],
    word_order=1
).score

chrfpp_score = sacrebleu.corpus_chrf(
    predictions,
    [reference],
    word_order=2
).score

# ------------------------------------------------------------
# 6. TER (Translation Edit Rate)
# ------------------------------------------------------------
ter_score = sacrebleu.corpus_ter(
    predictions,
    [reference]
).score

# ------------------------------------------------------------
# 7. METEOR
# ------------------------------------------------------------
meteor = evaluate.load("meteor")
meteor_score = meteor.compute(
    predictions=predictions,
    references=reference
)["meteor"]

# ------------------------------------------------------------
# 8. COMET / Indic-COMET (Neural Reference Metric)
# ------------------------------------------------------------
try:
    from comet import download_model, load_from_checkpoint
    model_path = download_model("Unbabel/wmt22-comet-da")
    comet_model = load_from_checkpoint(model_path)

    comet_input = [
        {"src": s, "mt": p, "ref": r}
        for s, p, r in zip(en_sents, predictions, reference)
    ]

    comet_score = comet_model.predict(
        comet_input,
        batch_size=8,
        gpus=1 if torch.cuda.is_available() else 0
    ).system_score
except Exception as e:
    print(f"COMET metric not computed ({e}). Skipping COMET.")
    comet_score = None

# ============================================================
# Print Summary
# ============================================================
print("\n" + "=" * 65)
print("     BHASHAVERSE EVALUATION RESULTS ON ANNOTATED DATASET")
print("=" * 65)
print(f"Dataset Sentences             : {len(predictions)}")
print("-" * 65)
print(f"SacreBLEU (Standard '13a')    : {sacrebleu_13a:.2f}")
print(f"SacreBLEU (Indic-Tokenized)   : {sacrebleu_indic:.2f}  <-- Recommended for Indic MT")
if sacrebleu_flores is not None:
    print(f"SacreBLEU (FLORES)            : {sacrebleu_flores:.2f}")
print(f"HuggingFace BLEU              : {bleu_score:.4f}")
print(f"chrF                          : {chrf_score:.2f}")
print(f"chrF++ (word_order=2)         : {chrfpp_score:.2f}  <-- Primary Morphological Metric")
print(f"METEOR                        : {meteor_score:.4f}")
print(f"TER (Lower is better)         : {ter_score:.2f}")
if comet_score is not None:
    print(f"COMET (wmt22-comet-da)        : {comet_score:.4f}")
# ============================================================
# Save Metrics to Files
# ============================================================
SAVE_DIR = os.path.dirname(PRED_CSV)
metrics_dict = {
    "Model": ["BhashaVerse (ltrciiith/bhashaverse)"],
    "Dataset": [os.path.basename(PRED_CSV)],
    "Sentence_Count": [len(predictions)],
    "SacreBLEU_13a": [round(sacrebleu_13a, 2)],
    "SacreBLEU_Indic_Tokenized": [round(sacrebleu_indic, 2)],
    "SacreBLEU_FLORES": [round(sacrebleu_flores, 2) if sacrebleu_flores is not None else None],
    "HF_BLEU": [round(bleu_score, 4)],
    "chrF": [round(chrf_score, 2)],
    "chrF++": [round(chrfpp_score, 2)],
    "METEOR": [round(meteor_score, 4)],
    "TER": [round(ter_score, 2)],
    "COMET": [round(comet_score, 4) if comet_score is not None else None],
}

metrics_df = pd.DataFrame(metrics_dict)
metrics_csv_path = os.path.join(SAVE_DIR, "bhashaverse_annotated_metrics.csv")
metrics_txt_path = os.path.join(SAVE_DIR, "bhashaverse_annotated_metrics.txt")

metrics_df.to_csv(metrics_csv_path, index=False)
metrics_df.to_csv("/content/bhashaverse_annotated_metrics.csv", index=False)

with open(metrics_txt_path, "w", encoding="utf-8") as f:
    for col in metrics_df.columns:
        f.write(f"{col}: {metrics_df[col][0]}\n")

print(f"\nEvaluation metrics saved to:\n1. {metrics_csv_path}\n2. {metrics_txt_path}\n3. /content/bhashaverse_annotated_metrics.csv")
display(metrics_df)


Loading predictions from: /content/drive/MyDrive/Inference final/BhashaVerse/Annotated_Filtered/predictions.csv
Evaluating 100 sentence pairs...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


...

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecat

COMET metric not computed (not enough values to unpack (expected 3, got 2)). Skipping COMET.

     BHASHAVERSE EVALUATION RESULTS ON ANNOTATED DATASET
Dataset Sentences             : 100
-----------------------------------------------------------------
SacreBLEU (Standard '13a')    : 4.15
SacreBLEU (Indic-Tokenized)   : 4.34  <-- Recommended for Indic MT
SacreBLEU (FLORES)            : 16.61
HuggingFace BLEU              : 0.0415
chrF                          : 42.88
chrF++ (word_order=2)         : 38.62  <-- Primary Morphological Metric
METEOR                        : 0.2667
TER (Lower is better)         : 87.30
-----------------------------------------------------------------
Multi-Reference Scores (Cleft + Canonical):
Multi-Ref SacreBLEU (13a)     : 4.72
Multi-Ref SacreBLEU (Indic)   : 4.87
Multi-Ref chrF++              : 39.80

Evaluation metrics saved to:
1. /content/drive/MyDrive/Inference final/BhashaVerse/Annotated_Filtered/bhashaverse_annotated_metrics.csv
2. /content/drive/My

,Model,Dataset,Sentence_Count,SacreBLEU_13a,SacreBLEU_Indic_Tokenized,SacreBLEU_FLORES,HF_BLEU,chrF,chrF++,METEOR,TER,COMET,MultiRef_SacreBLEU_13a,MultiRef_SacreBLEU_Indic,MultiRef_chrF++
0,BhashaVerse (ltrciiith/bhashaverse),predictions.csv,100,4.15,4.34,16.61,0.0415,42.88,38.62,0.2667,87.3,None,4.72,4.87,39.8


## Compute COMET Metric (Unbabel/wmt22-comet-da)
This cell installs `unbabel-comet`, evaluates BhashaVerse predictions against reference sentences using `Unbabel/wmt22-comet-da`, and permanently updates `bhashaverse_annotated_metrics.csv` and `bhashaverse_annotated_metrics.txt`.

In [ ]:
# ============================================================
# Compute COMET Metric (Unbabel/wmt22-comet-da) & Update Files
# ============================================================
%%capture
!pip install -q unbabel-comet

import os
import json
import torch
import pandas as pd
from comet import download_model, load_from_checkpoint

# 1. Locate Predictions File
pred_candidates = [
    "/content/drive/MyDrive/Inference/BhashaVerse_Results/bhashaverse_annotated_predictions.csv",
    "/content/drive/MyDrive/Inference/BhashaVerse_Results/predictions.csv",
    "/content/drive/MyDrive/Inference final/BhashaVerse/Annotated_Filtered/predictions.csv",
    "/content/drive/MyDrive/Inference/BhashaVerse/Annotated_Filtered/predictions.csv",
    os.path.join(os.getcwd(), "Inference", "BhashaVerse_Results", "bhashaverse_annotated_predictions.csv"),
    os.path.join(os.getcwd(), "Inference", "BhashaVerse_Results", "predictions.csv"),
    os.path.join(os.getcwd(), "Inference final", "BhashaVerse", "Annotated_Filtered", "predictions.csv"),
]

pred_csv = next((p for p in pred_candidates if os.path.exists(p)), None)

if pred_csv and os.path.exists(pred_csv):
    print(f"Loading predictions from: {pred_csv}")
    df_eval = pd.read_csv(pred_csv, encoding="utf-8-sig")
    en_col = next((c for c in df_eval.columns if "english" in c.lower()), df_eval.columns[0])
    ref_col = next((c for c in df_eval.columns if "malayalam" in c.lower() and "pred" not in c.lower() and "bhasha" not in c.lower()), df_eval.columns[1])
    mt_col = next((c for c in df_eval.columns if "pred" in c.lower() or "bhasha" in c.lower()), df_eval.columns[-1])

    src_list = df_eval[en_col].astype(str).str.strip().tolist()
    ref_list = df_eval[ref_col].astype(str).str.strip().tolist()
    mt_list = df_eval[mt_col].astype(str).str.strip().tolist()
elif "en_sents" in globals() and "predictions" in globals() and "reference" in globals():
    src_list = en_sents
    mt_list = predictions
    ref_list = reference
    pred_csv = "/content/drive/MyDrive/Inference/BhashaVerse_Results/bhashaverse_annotated_predictions.csv"
else:
    raise FileNotFoundError("Could not locate predictions CSV file to compute COMET.")

print(f"Evaluating {len(src_list)} sentence pairs with COMET...")

# 2. Load COMET model
print("Downloading and loading model: Unbabel/wmt22-comet-da...")
model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

# 3. Run Inference
comet_data = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(src_list, mt_list, ref_list)]
comet_output = comet_model.predict(comet_data, batch_size=8, gpus=1 if torch.cuda.is_available() else 0)
comet_score = round(float(comet_output.system_score), 4)

print("\n" + "=" * 60)
print(f"  BHASHAVERSE COMET SCORE (wmt22-comet-da): {comet_score:.4f}")
print("=" * 60)

# 4. Update Metrics CSV & TXT permanently across all result paths
save_dirs = [
    os.path.dirname(pred_csv) if pred_csv else None,
    "/content/drive/MyDrive/Inference/BhashaVerse_Results",
    "/content/drive/MyDrive/Inference final/BhashaVerse/Annotated_Filtered",
    "/content"
]

updated_df = None
for sdir in [d for d in save_dirs if d and os.path.exists(d)]:
    for base_m in ["bhashaverse_annotated_metrics", "metrics"]:
        m_csv = os.path.join(sdir, f"{base_m}.csv")
        m_txt = os.path.join(sdir, f"{base_m}.txt")
        if os.path.exists(m_csv):
            df_m = pd.read_csv(m_csv)
            df_m["COMET"] = comet_score
            df_m.to_csv(m_csv, index=False)
            updated_df = df_m
            print(f"Updated {m_csv}")
        if os.path.exists(m_txt):
            with open(m_txt, "r", encoding="utf-8") as f:
                lines = f.readlines()
            newlines = [l for l in lines if not l.startswith("COMET:")]
            newlines.append(f"COMET: {comet_score}\n")
            with open(m_txt, "w", encoding="utf-8") as f:
                f.writelines(newlines)
            print(f"Updated {m_txt}")

if updated_df is not None:
    display(updated_df)
